<style>
@page {
    size: A5;
    margin: 18mm 16mm 18mm 16mm;
}

body {
    font-family: "DejaVu Sans", "Arial", sans-serif;
    font-size: 12px;
    line-height: 1.35;
}

.jp-Notebook {
    max-width: 180mm;
    margin: 0 auto;
}

.jp-Cell {
    page-break-inside: avoid;
    break-inside: avoid;
}

pre, code {
    white-space: pre-wrap;
    word-break: break-word;
    overflow-wrap: anywhere;
}

table {
    width: 100%;
    table-layout: fixed;
    word-wrap: break-word;
}

img, svg, canvas {
    max-width: 100%;
    height: auto;
}

.output_area pre {
    white-space: pre-wrap;
    word-break: break-word;
    overflow-wrap: anywhere;
}
</style>

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy.stats as st
import seaborn as sns
import math as mt
from skimage.io import imread, imsave

np.random.seed(777)

In [2]:
n = 100
beta = 0.95

theta = 25 # <- захотели
X = st.uniform(loc=theta, scale=2*theta-theta).rvs(size=n)

In [3]:
Mx = np.mean(X)
x_max = np.max(X)

theta1 = 2/3 * Mx                       # <- оценка, полученная ОММ
theta2 = (n + 1) * x_max / (2*n + 1)    # <- исправленная оценка, полученная ОМП

**f)** Для сгенерированной выборки размера $ n = 100 $ и $ \theta = 25 $ вычислим доверительные интервалы

In [4]:
# Точный метод

d_left = x_max/(np.pow(1 + beta, 1/n) + 1)
d_right = x_max/(np.pow(1 - beta, 1/n) + 1)
accurate_l = d_right - d_left

print(f"Точный доверительный интервал для theta: {d_left:.5f} ≲ theta ≲ {d_right:.5f}")
print(f"Длина интервала l \u2248 {accurate_l:.5f}")

Точный доверительный интервал для theta: 24.91103 ≲ theta ≲ 25.36885
Длина интервала l ≈ 0.45782


In [5]:
# Асимптотический метод

u = st.norm.ppf((1 + beta)/2)

d_left = theta1 - u * theta1 / np.sqrt(27*n)
d_right = theta1 + u * theta1 / np.sqrt(27*n)
asymptotic_l = d_right - d_left

print(f"Асимптотический доверительный интервал для theta: {d_left:.5f} ≲ theta ≲ {d_right:.5f}")
print(f"Длина интервала l ≈ {asymptotic_l:.5f}")

Асимптотический доверительный интервал для theta: 23.82722 ≲ theta ≲ 25.69518
Длина интервала l ≈ 1.86796


**g)** Численно построить бутстраповский интервал

In [6]:
N = 1000

def omp_bootstrap(x, N):
    arr = list()
    n = len(x)

    for _ in range(N):
        sample = np.random.choice(x, size=n, replace=True)        
        arr += [(n+1) * np.max(sample) / (2*n + 1) - theta2]

    return sorted(np.array(arr))

bootstrap_array = omp_bootstrap(X, N)

t1 = int((1 - beta)/2 * N - 1)
t2 = int((1 + beta)/2 * N - 1)

bootstrap_left = theta1 - bootstrap_array[t2]
bootstrap_right = theta1 - bootstrap_array[t1]
bootstrap_l_omp = bootstrap_right - bootstrap_left

print(f"Непараметрический бутстраповский доверительный интервал для theta (ОМП): {bootstrap_left:.5f} ≲ theta ≲ {bootstrap_right:.5f}")
print(f"Длина интервала l \u2248 {bootstrap_l_omp:.5f}")

Непараметрический бутстраповский доверительный интервал для theta (ОМП): 24.76120 ≲ theta ≲ 25.19219
Длина интервала l ≈ 0.43099


In [7]:
N = 1000

def omm_bootstrap(x, N):
    arr = list()
    n = len(x)

    for _ in range(N):
        sample = np.random.choice(x, size=n, replace=True)        
        arr += [2/3 * np.mean(sample) - theta1]

    return sorted(np.array(arr))

bootstrap_array = omm_bootstrap(X, N)

t1 = int((1 - beta)/2 * N - 1)
t2 = int((1 + beta)/2 * N - 1)

bootstrap_left = theta1 - bootstrap_array[t2]
bootstrap_right = theta1 - bootstrap_array[t1]
bootstrap_l_omm = bootstrap_right - bootstrap_left

print(f"Непараметрический бутстраповский доверительный интервал для theta (ОММ): {bootstrap_left:.5f} ≲ theta ≲ {bootstrap_right:.5f}")
print(f"Длина интервала l \u2248 {bootstrap_l_omm:.5f}")

Непараметрический бутстраповский доверительный интервал для theta (ОММ): 23.86801 ≲ theta ≲ 25.65124
Длина интервала l ≈ 1.78323


**h)** Сравнить все интервалы

In [8]:
from tabulate import tabulate

lengths = {
    "Точный": accurate_l,
    "Асимптотический": asymptotic_l,
    "Бутстрап ОММ": bootstrap_l_omm,
    "Бутстрап ОМП": bootstrap_l_omp
}

sorted_lengths = sorted([(value, name) for name, value in lengths.items()])

table_data = []
for i, (length, name) in enumerate(sorted_lengths, start=1):
    table_data.append([i, name, np.round(length, 3)])

print(tabulate(table_data, headers=["№", "Метод", "Длина (l)"], tablefmt="fancy_outline"))

╒═════╤═════════════════╤═════════════╕
│   № │ Метод           │   Длина (l) │
╞═════╪═════════════════╪═════════════╡
│   1 │ Бутстрап ОМП    │       0.431 │
│   2 │ Точный          │       0.458 │
│   3 │ Бутстрап ОММ    │       1.783 │
│   4 │ Асимптотический │       1.868 │
╘═════╧═════════════════╧═════════════╛
